# SC-8-Account-Abstraction - ERC-4337

**Navigation** : [Index](../README.md) | [<< DAO](SC-7-DAO-Governance.ipynb) | [LLM >>](SC-8b-LLM-Assisted.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'**account abstraction** (ERC-4337)
2. Creer des **UserOperations**
3. Implementer un **Smart Account** simple
4. Comprendre les **Paymasters**

### Duree estimee : 50 minutes

---

## 1. Concepts ERC-4337

ERC-4337 permet de separer la logique des comptes des wallets externes.

| Composant | Description |
|-----------|-------------|
| **UserOperation** | Transaction utilisateur (comme transfer, swap) |
| **EntryPoint** | Contrat singleton qui traite les UserOps |
| **Smart Account** | Wallet intelligent (logique metier en contrat) |
| **Paymaster** | Sponsor de frais de gas |
| **Bundler** | Empaquete les UserOps en batch |

In [ ]:
# UserOperation structure
USER_OP_STRUCT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

struct UserOperation {
    address sender;
    uint256 nonce;
    bytes initCode;
    bytes callData;
    uint256 callGasLimit;
    uint256 verificationGasLimit;
    uint256 preVerificationGas;
    uint256 maxFeePerGas;
    uint256 maxPriorityFeePerGas;
    bytes paymasterAndData;
    bytes signature;
}
'''

print("Structure UserOperation:")
print(USER_OP_STRUCT)

---

## 2. Smart Account Simple

Un smart account est un contrat qui peut executer des transactions.

In [ ]:
# Simple Smart Account
SIMPLE_ACCOUNT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@account-abstraction/contracts/core/BaseAccount.sol";
import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";

contract SimpleAccount is BaseAccount {
    using ECDSA for bytes32;

    address public owner;
    IEntryPoint private immutable _entryPoint;

    event SimpleAccountInitialized(IEntryPoint indexed entryPoint, address indexed owner);

    constructor(address anOwner, IEntryPoint anEntryPoint) {
        owner = anOwner;
        _entryPoint = anEntryPoint;
    }

    function entryPoint() public view override returns (IEntryPoint) {
        return _entryPoint;
    }

    // Initialisation
    function _initialize(address anOwner) internal {
        owner = anOwner;
        emit SimpleAccountInitialized(_entryPoint, anOwner);
    }

    // Verification de signature
    function _validateSignature(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash
    ) internal virtual override returns (uint256 validationData) {
        bytes32 hash = MessageHashUtils.toEthSignedMessageHash(userOpHash);
        if (owner != ECDSA.recover(hash, userOp.signature)) {
            return SIG_VALIDATION_FAILED;
        }
        return SIG_VALIDATION_SUCCESS;
    }

    // Executer une transaction
    function execute(address dest, uint256 value, bytes calldata func) external {
        _requireFromEntryPointOrOwner();
        (bool success, ) = dest.call{value: value}(func);
        require(success, "call failed");
    }

    // Executer un batch de transactions
    function executeBatch(
        address[] calldata dest,
        uint256[] calldata value,
        bytes[] calldata func
    ) external {
        _requireFromEntryPointOrOwner();
        require(dest.length == func.length && dest.length == value.length, "wrong array lengths");
        for (uint256 i = 0; i < dest.length; i++) {
            (bool success, ) = dest[i].call{value: value[i]}(func[i]);
            require(success, "call failed");
        }
    }

    function _requireFromEntryPointOrOwner() internal view {
        require(
            msg.sender == address(_entryPoint) || msg.sender == owner,
            "only entry point or owner"
        );
    }
}
'''

print("Smart Account:")
print(SIMPLE_ACCOUNT)

---

## 3. Paymaster

Un paymaster peut payer les frais de gas pour les utilisateurs.

In [ ]:
# Verifying Paymaster
PAYMASTER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@account-abstraction/contracts/core/BasePaymaster.sol";

contract VerifyingPaymaster is BasePaymaster {
    using ECDSA for bytes32;

    address public verifyingSigner;

    constructor(IEntryPoint _entryPoint, address _verifyingSigner)
        BasePaymaster(_entryPoint)
    {
        verifyingSigner = _verifyingSigner;
    }

    // Pack userOp.hash et validUntil/validAfter
    function getHash(
        PackedUserOperation calldata userOp,
        uint48 validUntil,
        uint48 validAfter
    ) public pure returns (bytes32) {
        return keccak256(abi.encode(
            hash(userOp),
            validUntil,
            validAfter
        ));
    }

    // Validation du paymaster
    function _validatePaymasterUserOp(
        PackedUserOperation calldata userOp,
        bytes32,
        uint256 requiredPreFund
    ) internal override returns (bytes memory context, uint256 validationData) {
        // Decode paymasterAndData: [20 bytes paymaster] + [6 bytes validUntil] + [6 bytes validAfter] + [signature]
        (uint48 validUntil, uint48 validAfter, bytes calldata signature) = 
            parsePaymasterAndData(userOp.paymasterAndData);

        // Verifier la signature
        bytes32 hash = getHash(userOp, validUntil, validAfter);
        bytes32 ethSignedHash = MessageHashUtils.toEthSignedMessageHash(hash);

        if (verifyingSigner != ECDSA.recover(ethSignedHash, signature)) {
            return ("", _packValidationData(true, validUntil, validAfter));
        }

        return ("", _packValidationData(false, validUntil, validAfter));
    }

    function parsePaymasterAndData(bytes calldata paymasterAndData)
        internal pure
        returns (uint48 validUntil, uint48 validAfter, bytes calldata signature)
    {
        require(paymasterAndData.length >= 52, "invalid paymasterAndData");
        validUntil = uint48(bytes6(paymasterAndData[20:26]));
        validAfter = uint48(bytes6(paymasterAndData[26:32]));
        signature = paymasterAndData[32:];
    }
}
'''

print("Verifying Paymaster:")
print(PAYMASTER)

---

## 4. Flux de Transaction ERC-4337

In [ ]:
# Flux ERC-4337
print("""
FLUX DE TRANSACTION ERC-4337

1. CREATION
   User -> Crée UserOperation -> Signe avec clé privée

2. SOUMISSION
   UserOp -> Bundler (via RPC ou API)

3. VALIDATION
   Bundler -> simulateValidation() sur EntryPoint
   EntryPoint -> appelle validateUserOp() sur Smart Account
   Smart Account -> vérifie signature

4. EXECUTION
   Bundler -> agrège plusieurs UserOps
   Bundler -> handleOps() sur EntryPoint
   EntryPoint -> execute les opérations validées

AVANTAGES:
- Wallets sans seed phrase (social recovery)
- Gas sponsoring (paymasters)
- Batch transactions
- Transactions sans ETH (pay with tokens)
- Logic programmable (2FA, spending limits)

ADDRESSES EntryPoint:
- Mainnet: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Sepolia: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Goerli: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
""")

---

## 5. Exercices

In [ ]:
# Exercice: Social Recovery
EXERCISE_RECOVERY = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@account-abstraction/contracts/core/BaseAccount.sol";

contract SocialRecoveryAccount is BaseAccount {
    address public owner;
    address[] public guardians;
    mapping(address => bool) public isGuardian;
    uint256 public guardianCount;
    uint256 public threshold;  // nombre de gardiens requis

    struct RecoveryRequest {
        address newOwner;
        uint256 approvals;
        mapping(address => bool) hasApproved;
    }

    RecoveryRequest public pendingRecovery;

    event GuardianAdded(address indexed guardian);
    event RecoveryStarted(address indexed newOwner);
    event RecoveryCompleted(address indexed newOwner);

    constructor(
        address _owner,
        address[] memory _guardians,
        uint256 _threshold,
        IEntryPoint _entryPoint
    ) BaseAccount(_entryPoint) {
        owner = _owner;
        threshold = _threshold;
        for (uint256 i = 0; i < _guardians.length; i++) {
            _addGuardian(_guardians[i]);
        }
    }

    function _addGuardian(address guardian) internal {
        require(!isGuardian[guardian], "Already guardian");
        guardians.push(guardian);
        isGuardian[guardian] = true;
        guardianCount++;
        emit GuardianAdded(guardian);
    }

    // Demander une recuperation
    function startRecovery(address newOwner) external {
        require(isGuardian[msg.sender], "Not guardian");
        require(pendingRecovery.newOwner == address(0), "Recovery pending");

        pendingRecovery.newOwner = newOwner;
        pendingRecovery.approvals = 1;
        pendingRecovery.hasApproved[msg.sender] = true;

        emit RecoveryStarted(newOwner);
    }

    // Approuver la recuperation
    function approveRecovery() external {
        require(isGuardian[msg.sender], "Not guardian");
        require(pendingRecovery.newOwner != address(0), "No recovery pending");
        require(!pendingRecovery.hasApproved[msg.sender], "Already approved");

        pendingRecovery.approvals++;
        pendingRecovery.hasApproved[msg.sender] = true;

        if (pendingRecovery.approvals >= threshold) {
            owner = pendingRecovery.newOwner;
            delete pendingRecovery;
            emit RecoveryCompleted(owner);
        }
    }

    function _validateSignature(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash
    ) internal override returns (uint256) {
        bytes32 hash = MessageHashUtils.toEthSignedMessageHash(userOpHash);
        if (owner != ECDSA.recover(hash, userOp.signature)) {
            return SIG_VALIDATION_FAILED;
        }
        return SIG_VALIDATION_SUCCESS;
    }
}
'''

print("Exercice Social Recovery:")
print(EXERCISE_RECOVERY)

---

## 6. Resume

| Concept | Description |
|---------|-------------|
| UserOperation | Structure de transaction ERC-4337 |
| EntryPoint | Singleton qui traite les UserOps |
| Smart Account | Wallet programmable |
| Paymaster | Sponsor de gas fees |
| Bundler | Empaqueteur de transactions |
| Social Recovery | Recuperation via gardiens |

---

**Notebook suivant** : [SC-8b-LLM-Assisted](SC-8b-LLM-Assisted.ipynb)